# Runtime 안정성 분석

## 분석 목적
모델 품질과 분리하여 latency, 실패 및 token 관측 범위를 분석한다.

## 가설
H0: 동일 Case에서 full-response latency의 모델 간 paired 분포 차이가 없다.

## 사용할 변수
measurement_type, full_response_latency_millis, attempt_elapsed_millis, initial_runtime_latency_millis, provider_http_status, provider_status, token_usage_status와 token 수

## 통계기법 선택 이유
타입별 mean/median/P95/P99 및 성공·실패 strata. MOCK 0과 attempt는 실제 full response와 합치지 않는다. 2모델은 독립 Case 및 대칭 차이 확인 시 Wilcoxon+rank-biserial. 3모델 이상 Friedman은 근사 유효성 조건 충족 시에만 실행하며 Kendall W를 표시한다.

## 해석 기준
완전한 paired Case 수와 제외 수를 표시한다. 작은 표본의 P99는 SLA가 아니다. NOT_ATTEMPTED와 결측 token은 실패/0 token으로 일괄 치환하지 않는다. TRANSPORT에는 timeout 외 오류도 있으므로 timeout 비율은 평가 불가.

기본 입력은 **합성 Consumer 시험 fixture**이다. 실제 Bundle은 `ADP_AI_BUNDLE_SOURCE`로 지정한다. Case 독립성/대칭성은 자동 추정하지 않는다.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "02_ai/src").is_dir())
sys.path.insert(0, str(ROOT / "02_ai/src"))
from adp_da.bundle_analysis import analyze_bundle  # noqa: E402
from adp_da.bundle_dataset import execution_dataframe  # noqa: E402
from adp_da.bundle_loader import load_bundle  # noqa: E402

source = os.environ.get("ADP_AI_BUNDLE_SOURCE")
synthetic = source is None or os.environ.get("ADP_AI_SYNTHETIC") == "1"
if source is None:
    source = str(ROOT / "02_ai/tests/fixtures/evaluation_bundle.synthetic.json")
bundle, metadata = load_bundle(
    source, ROOT / "02_ai/data/interim/ai_evaluation/raw",
    evaluation_run_id=os.environ.get("ADP_AI_EVALUATION_RUN_ID"),
    token=os.environ.get("ADP_BE_TOKEN"),
    remote_bearer_enabled=os.environ.get("ADP_BE_REMOTE_BEARER_AUTH_ENABLED") == "YES",
    local_admin_user_id=os.environ.get("ADP_BE_LOCAL_ADMIN_USER_ID"),
    local_admin_roles=os.environ.get("ADP_BE_LOCAL_ADMIN_ROLES"),
)
frame = execution_dataframe(bundle)
artifacts = analyze_bundle(
    frame, synthetic=synthetic,
    independent_cases=os.environ.get("ADP_AI_INDEPENDENT_CASES") == "1",
    symmetric_differences=os.environ.get("ADP_AI_SYMMETRIC_DIFFERENCES") == "1",
)
print("SYNTHETIC FIXTURE — SOFTWARE VALIDATION ONLY" if synthetic else "USER-SUPPLIED BUNDLE")
display({k: bundle["manifest"][k] for k in ("bundle_id", "content_digest", "execution_count")})


In [ ]:
display(artifacts["runtime_analysis"])
display(artifacts["failure_analysis"])
